In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
import gradio as gr

In [3]:
path = "data" 
rows = []
labels = []

folders = sorted(os.listdir(path))
for folder in folders:
    folder_path = os.path.join(path, folder)
    if os.path.isdir(folder_path):
        imgs = os.listdir(folder_path)
        for name in imgs:
            img_path = os.path.join(folder_path, name)
            img = cv2.imread(img_path, 0)
            if img is not None:
                img_28 = cv2.resize(img, (28, 28))
                img_1d = img_28.flatten()
                rows.append(img_1d)
                labels.append(folder)

In [4]:
df1 = pd.DataFrame(rows, dtype=np.uint8)
df2 = pd.DataFrame(labels, columns=["label"])
df = pd.concat([df1, df2], axis=1)

In [5]:
X = df.drop(columns=["label"])
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

,n_neighbors,3
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [12]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [7]:
pred_knn = knn.predict(X_test)
pred_dt = dt.predict(X_test)

acc_knn = accuracy_score(y_test, pred_knn)
acc_dt = accuracy_score(y_test, pred_dt)

print(f"KNN Accuracy: {acc_knn*100:.2f}%")
print(f"DT Accuracy:  {acc_dt*100:.2f}%")

KNN Accuracy: 75.40%
DT Accuracy:  75.86%


In [16]:
best_model = dt 
 
with open("mask_model.pkl", "wb") as f: 
    pickle.dump(best_model, f)

In [17]:
def predict_mask(img_path):
    with open("mask_model.pkl", "rb") as f:
        model = pickle.load(f)
    img = cv2.imread(img_path, 0)
    if img is None:
        return "Image not found!"
    img_28 = cv2.resize(img, (28, 28))
    img_1d = img_28.flatten().reshape(1, -1)
    res = model.predict(img_1d)
    return res[0]

def gradio_predict(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_28 = cv2.resize(gray, (28, 28))
    img_1d = img_28.flatten().reshape(1, -1)
    with open("mask_model.pkl", "rb") as f:
        model = pickle.load(f)
    res = model.predict(img_1d)
    return f"Prediction: {res[0]}"




In [18]:
app = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(label="Upload Face Image"),
    outputs=gr.Textbox(label="Mask Status"),
    title="😷 Face Mask Detector"
)

In [19]:
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
